In [1]:
!pip install sagemaker==2.224.1 xgboost scikit-learn -q

In [2]:
import boto3
import json
import sagemaker
import pandas as pd
import tarfile
from sagemaker.workflow.pipeline_context import PipelineSession

sagemaker_session    = sagemaker.session.Session()
pipeline_session     = PipelineSession()
region               = sagemaker_session.boto_region_name
role                 = sagemaker.get_execution_role()
default_bucket       = sagemaker_session.default_bucket()
pipeline_name        = "sagemaker-mlops-train-pipeline"
model_package_group_name = "BankModelPackageGroup"

print("Region:", region)
print("Bucket:", default_bucket)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region: us-east-1
Bucket: sagemaker-us-east-1-928747699677


In [3]:

from sagemaker.workflow.parameters import (
    ParameterInteger, ParameterString, ParameterFloat
)

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount", default_value=1
)
processing_instance_type = ParameterString(
    name="ProcessingInstanceType", default_value="ml.t3.medium"
)
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
input_data = ParameterString(
    name="InputData",
    default_value=f"s3://{sagemaker_session.default_bucket()}/raw/bank-full.csv"
)
auc_score_threshold = ParameterFloat(
    name="AucScoreThreshold", default_value=0.75
)

In [4]:
# !wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
# !unzip bank+marketing.zip

In [5]:
# !unzip bank.zip

In [6]:
boto3.client("s3").upload_file(
    "bank-full.csv", default_bucket, "raw/bank-full.csv"
)
print(f"✅ Uploaded to s3://{default_bucket}/raw/bank-full.csv")

✅ Uploaded to s3://sagemaker-us-east-1-928747699677/raw/bank-full.csv


In [7]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

sklearn_processor = SKLearnProcessor(
    framework_version="1.0-1",
    instance_type="ml.t3.medium",
    instance_count=processing_instance_count,
    base_job_name="sklearn-bank-process",
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = sklearn_processor.run(
    inputs=[
        ProcessingInput(source=input_data, destination="/opt/ml/processing/input")
    ],
    outputs=[
        ProcessingOutput(output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{default_bucket}/output/train"),
        ProcessingOutput(output_name="validation",
            source="/opt/ml/processing/validation",
            destination=f"s3://{default_bucket}/output/validation"),
        ProcessingOutput(output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{default_bucket}/output/test"),
    ],
    code="preprocess-bank.py",
)

step_process = ProcessingStep(name="BankModelProcess", step_args=processor_args)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [8]:
from xgboost import XGBClassifier
import tarfile, boto3

df_train = pd.read_csv(f"s3://{default_bucket}/output/train/train.csv", header=None)
df_val   = pd.read_csv(f"s3://{default_bucket}/output/validation/validation.csv", header=None)

X_train, y_train = df_train.iloc[:, 1:], df_train.iloc[:, 0]
X_val,   y_val   = df_val.iloc[:, 1:],   df_val.iloc[:, 0]

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print("✅ Training done")

# ✅ Save as explicit JSON — no ambiguity
model.save_model("xgboost-model.json")

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("xgboost-model.json", arcname="xgboost-model.json")  # ← .json

boto3.client("s3").upload_file(
    "model.tar.gz", default_bucket, "output/pretrained/model.tar.gz"
)
pretrained_model_uri = f"s3://{default_bucket}/output/pretrained/model.tar.gz"
print("✅ Uploaded:", pretrained_model_uri)

/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [11:10:40] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✅ Training done
✅ Uploaded: s3://sagemaker-us-east-1-928747699677/output/pretrained/model.tar.gz


In [9]:
image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.5-1",
    py_version="py3",
    instance_type="ml.t3.medium",
)

In [10]:
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.properties import PropertyFile

script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name="script-bank-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=pretrained_model_uri,
            destination="/opt/ml/processing/model"
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=f"s3://{default_bucket}/output/evaluation"
        ),
    ],
    code="evaluation-bank.py",
)

evaluation_report = PropertyFile(
    name="BankEvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_eval = ProcessingStep(
    name="BankEvalModel",
    step_args=eval_args,
    property_files=[evaluation_report],
)

/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [11]:
from sagemaker import Model
from sagemaker.workflow.model_step import ModelStep
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model = Model(
    image_uri=image_uri,
    model_data=pretrained_model_uri,
    sagemaker_session=pipeline_session,
    role=role,
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)

step_register = ModelStep(name="BankRegisterModel", step_args=register_args)

In [12]:
from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

cond_lte = ConditionGreaterThan(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.auc_score.value",
    ),
    right=auc_score_threshold,
)

step_cond = ConditionStep(
    name="CheckAUCScoreBankEvaluation",
    conditions=[cond_lte],
    if_steps=[step_register],
)

In [13]:
import json
from sagemaker.workflow.pipeline import Pipeline

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        processing_instance_type,
        model_approval_status,
        input_data,
        auc_score_threshold,
    ],
    steps=[step_process, step_eval, step_cond],  # no training step
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()
print("✅ Pipeline started:", execution.arn)

✅ Pipeline started: arn:aws:sagemaker:us-east-1:928747699677:pipeline/sagemaker-mlops-train-pipeline/execution/nyvxwgqlwyqu


In [14]:
# import boto3

# sm = boto3.client("sagemaker", region_name=region)

# # Get latest execution
# executions = sm.list_pipeline_executions(
#     PipelineName="sagemaker-mlops-train-pipeline",
#     SortOrder="Descending"
# )
# execution_arn = executions["PipelineExecutionSummaries"][0]["PipelineExecutionArn"]
# print("Execution ARN:", execution_arn)

# # Get step details
# steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn)
# for step in steps["PipelineExecutionSteps"]:
#     print(f"\nStep: {step['StepName']}")
#     print(f"  Status: {step['StepStatus']}")
#     if "FailureReason" in step:
#         print(f"  Failure: {step['FailureReason']}")

In [15]:
# # Get the actual tuning job name from the step metadata
# for step in steps["PipelineExecutionSteps"]:
#     if step["StepName"] == "BankHyperParameterTuning":
#         print("Full step details:")
#         print(step)

In [16]:

# # List training jobs under the tuning job
# tuning_jobs = sm.list_hyper_parameter_tuning_jobs(
#     SortBy="CreationTime",
#     SortOrder="Descending"
# )
# if tuning_jobs["HyperParameterTuningJobSummaries"]:
#     tuning_name = tuning_jobs["HyperParameterTuningJobSummaries"][0]["HyperParameterTuningJobName"]
#     print("Tuning job name:", tuning_name)
    
#     detail = sm.describe_hyper_parameter_tuning_job(
#         HyperParameterTuningJobName=tuning_name
#     )
#     print("Status:", detail["HyperParameterTuningJobStatus"])
#     print("Failure reason:", detail.get("FailureReason", "None"))
    
#     # Get individual training jobs
#     training_jobs = sm.list_training_jobs_for_hyper_parameter_tuning_job(
#         HyperParameterTuningJobName=tuning_name
#     )
#     for tj in training_jobs["TrainingJobSummaries"]:
#         print(f"\nTraining job: {tj['TrainingJobName']}")
#         print(f"  Status: {tj['TrainingJobStatus']}")
#         if "FailureReason" in tj:
#             print(f"  Failure: {tj['FailureReason']}")

In [17]:
# import boto3

# sq = boto3.client("service-quotas", region_name="us-east-1")

# # List SageMaker training job quotas
# response = sq.list_service_quotas(ServiceCode="sagemaker")

# print("Available training instance quotas:")
# for q in response["Quotas"]:
#     if "training" in q["QuotaName"].lower() and q["Value"] > 0:
#         print(f"  {q['QuotaName']}: {q['Value']}")

In [18]:
with open("evaluation-bank.py", 'r') as f:
    print(f.read())

script = """
import json
import pathlib
import tarfile
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_curve, auc

if __name__ == "__main__":

    model_path = "/opt/ml/processing/model/model.tar.gz"
    with tarfile.open(model_path) as tar:
        tar.extractall(path="/opt/ml/processing/model/")
        print("Extracted:", tar.getnames())

    model = xgb.Booster()
    model.load_model("/opt/ml/processing/model/xgboost-model.json")
    print("Model loaded")

    df = pd.read_csv("/opt/ml/processing/test/test.csv", header=None)
    y_test = df.iloc[:, 0].to_numpy()
    X_test = xgb.DMatrix(df.iloc[:, 1:])

    predictions = model.predict(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, predictions)
    auc_score = float(auc(fpr, tpr))
    print(f"AUC Score: {auc_score:.4f}")

    report_dict = {
        "classification_metrics": {
            "auc_score": {
                "value": auc_score,
            },
        },
    }

    output

In [19]:
# import boto3

# sm = boto3.client("sagemaker", region_name=region)

# # Get latest execution steps
# executions = sm.list_pipeline_executions(
#     PipelineName="sagemaker-mlops-train-pipeline",
#     SortOrder="Descending"
# )
# execution_arn = executions["PipelineExecutionSummaries"][0]["PipelineExecutionArn"]

# steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn)
# for step in steps["PipelineExecutionSteps"]:
#     print(f"Step: {step['StepName']} — {step['StepStatus']}")
#     if "FailureReason" in step:
#         print(f"  Failure: {step['FailureReason']}")
#     if "Metadata" in step and step["Metadata"]:
#         print(f"  Metadata: {step['Metadata']}")

In [20]:
# import boto3

# log_client = boto3.client("logs", region_name="us-east-1")

# log_group = "/aws/sagemaker/ProcessingJobs"
# stream_name = "pipelines-ci4uu988igol-BankEvalModel-Hn00yfYnDu/algo-1-1749286"

# try:
#     streams = log_client.describe_log_streams(
#         logGroupName=log_group,
#         logStreamNamePrefix="pipelines-ci4uu988igol-BankEvalModel-Hn00yfYnDu"
#     )
#     print("Log streams found:")
#     for s in streams["logStreams"]:
#         print(" ", s["logStreamName"])

#     stream = streams["logStreams"][0]["logStreamName"]
#     events = log_client.get_log_events(
#         logGroupName=log_group,
#         logStreamName=stream,
#         limit=50
#     )
#     print("\n--- LOGS ---")
#     for e in events["events"]:
#         print(e["message"])
# except Exception as ex:
#     print("Error:", ex)

In [21]:
# import tarfile

# with tarfile.open("model.tar.gz", "r:gz") as tar:
#     print("Files in tar:", tar.getnames())

In [22]:
# with open("evaluation-bank.py", "r") as f:
#     content = f.read()
#     if '_name' in content and 'main_' in content:
#         print("✅ Script saved correctly with double underscores")
#     else:
#         print("❌ Still wrong — check manually")

In [23]:
# pipeline.upsert(role_arn=role)
# execution = pipeline.start()
# print("Started:", execution.arn)

In [24]:
# import boto3

# log_client = boto3.client("logs", region_name="us-east-1")
# log_group = "/aws/sagemaker/ProcessingJobs"

# # Get latest eval job name
# sm = boto3.client("sagemaker", region_name="us-east-1")
# jobs = sm.list_processing_jobs(
#     NameContains="BankEvalModel",
#     SortBy="CreationTime",
#     SortOrder="Descending"
# )
# job_name = jobs["ProcessingJobSummaries"][0]["ProcessingJobName"]
# print("Job:", job_name)

# # Get LAST log events — not first
# streams = log_client.describe_log_streams(
#     logGroupName=log_group,
#     logStreamNamePrefix=job_name
# )
# stream = streams["logStreams"][0]["logStreamName"]

# events = log_client.get_log_events(
#     logGroupName=log_group,
#     logStreamName=stream,
#     limit=50,
#     startFromHead=False    # ← get LAST events
# )
# print("\n--- LAST LOGS ---")
# for e in events["events"]:
#     print(e["message"])

In [25]:
import json
import pathlib
import tarfile
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_curve, auc
import boto3, os

# Download files locally to test
boto3.client("s3").download_file(
    default_bucket, "output/pretrained/model.tar.gz", "test_model.tar.gz"
)
boto3.client("s3").download_file(
    default_bucket, "output/test/test.csv", "test_data.csv"
)
print("✅ Downloaded files")

# Step 1: Extract
with tarfile.open("test_model.tar.gz") as tar:
    tar.extractall(path="./test_model/")
    print("Extracted:", tar.getnames())

# Step 2: Load model
print("Files in test_model/:", os.listdir("./test_model/"))
model = xgb.Booster()
model.load_model("./test_model/xgboost-model.json")
print("✅ Model loaded")

# Step 3: Load test data
df = pd.read_csv("test_data.csv", header=None)
print(f"Test shape: {df.shape}")
print(f"First row: {df.iloc[0].tolist()}")

y_test = df.iloc[:, 0].to_numpy()
X_test = xgb.DMatrix(df.iloc[:, 1:])

# Step 4: Predict
predictions = model.predict(X_test)
print(f"Predictions sample: {predictions[:5]}")

# Step 5: AUC
fpr, tpr, thresholds = roc_curve(y_test, predictions)
auc_score = float(auc(fpr, tpr))
print(f"✅ AUC Score: {auc_score:.4f}")

# Step 6: Write report
os.makedirs("./test_output/", exist_ok=True)
report_dict = {
    "classification_metrics": {
        "auc_score": {"value": auc_score}
    }
}
with open("./test_output/evaluation.json", "w") as f:
    f.write(json.dumps(report_dict))

print("✅ evaluation.json written successfully")
print("Contents:", json.dumps(report_dict))

✅ Downloaded files
Extracted: ['xgboost-model.json']
Files in test_model/: ['xgboost-model.json']
✅ Model loaded
Test shape: (6782, 52)
First row: [0.0, 60.0, -70.0, 13.0, 65.0, 2.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
Predictions sample: [2.1088365e-04 1.9876177e-04 5.4110349e-03 2.2513865e-02 4.6804073e-01]
✅ AUC Score: 0.9362
✅ evaluation.json written successfully
Contents: {"classification_metrics": {"auc_score": {"value": 0.9361677929476309}}}


/tmp/ipykernel_4687/3277374114.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="./test_model/")
